# OtiLLM v2.0 — JAIR Publication Evaluation
Run on Google Colab with **A100 GPU + High RAM**. This notebook is for the frozen JAIR rebuild only.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,2) if torch.cuda.is_available() else 0)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/OtiLLM_JAIR_v2')
for folder in ['manifest','clean_results','stress_results','scored_results','statistics','figures','logs']:
    (BASE/folder).mkdir(parents=True, exist_ok=True)
print('Drive root:', BASE)


In [ ]:
%cd /content
!rm -rf OtiLLM
!git clone --branch jair-v2.0-rebuild https://github.com/OtiEdema/OtiLLM.git
%cd /content/OtiLLM
!git rev-parse HEAD


In [ ]:
!python -m pip install -q -e '.[dev]'
!python -m pip install -q 'datasets==2.21.0' 'transformers==4.57.6' 'huggingface-hub==0.36.2' 'fsspec==2024.6.1' 'accelerate>=1.0,<2' 'scipy>=1.11'


## Pinned Qwen3-8B loading smoke test
This verifies that the exact publication backend fits the assigned A100 before any benchmark run.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = 'Qwen/Qwen3-8B'
REVISION = 'a0809d1efaf104a46bbc2acaecd69617e9696312'
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype='auto', device_map='auto')
print('Loaded:', MODEL, REVISION)
print('Device:', model.device)


In [ ]:
!nvidia-smi


## Frozen manifest
Do not start the publication run until the validated frozen manifest artifact has completed successfully on GitHub. Copy the final `jair_manifest.jsonl` and `jair_manifest.metadata.json` into `MyDrive/OtiLLM_JAIR_v2/manifest/`, then run the verification cell below.


In [ ]:
MANIFEST = BASE/'manifest'/'jair_manifest.jsonl'
META = BASE/'manifest'/'jair_manifest.metadata.json'
print('manifest exists:', MANIFEST.exists())
print('metadata exists:', META.exists())
if MANIFEST.exists():
    !python experiments/jair_v2_rebuild/validate_manifest.py "$MANIFEST" --expected-content-n 200 --require-full-xstest


## Clean evaluation
Run only after the manifest validation above succeeds. Results checkpoint directly to Google Drive and can resume after a Colab disconnect.


In [ ]:
CLEAN_OUT = BASE/'clean_results'/'raw_clean.jsonl'
print('Clean output:', CLEAN_OUT)
# Uncomment only after the frozen manifest has validated.
# !python experiments/jair_v2_rebuild/run_experiments.py --manifest "$MANIFEST" --systems experiments/jair_v2_rebuild/configs/systems_publication.json --backend experiments/jair_v2_rebuild/configs/backend.json --out "$CLEAN_OUT" --resume


## Retrieval-stress evaluation
This phase uses only retrieval-consuming systems and excludes XSTest automatically.


In [ ]:
STRESS_OUT = BASE/'stress_results'/'raw_stress.jsonl'
print('Stress output:', STRESS_OUT)
# Uncomment after the clean run is secured.
# !python experiments/jair_v2_rebuild/run_stress.py --manifest "$MANIFEST" --systems experiments/jair_v2_rebuild/configs/systems_stress.json --backend experiments/jair_v2_rebuild/configs/backend.json --out "$STRESS_OUT" --resume
